# MGS-2 : Composition de metaheuristiques -- Match et primitives de controle

**Navigation** : [Index](README.md) | [<< MGS-1 Introduction](MGS-1-Introduction.ipynb) | [MGS-3 Eukaryote >>](MGS-3-Eukaryote.ipynb)

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
1. **Configurer** le mecanisme de match (`MatchMetaHeuristic`, `MatchingKind`) pour controler la selection des parents lors du crossover
2. **Utiliser** les primitives de controle de flux (`SwitchMetaHeuristic`, `SizeBasedMetaHeuristic`, `GenerationMetaHeuristic`, `StageSwitchMetaHeuristic`) pour adapter le comportement en cours d'evolution
3. **Composer** ces primitives en strategies multi-phases qui combinent plusieurs comportements
4. **Comparer** les performances de differentes strategies sur un meme probleme d'optimisation

### Prerequis
- MGS-1 : Introduction a MetaGeneticSharp et au moteur autonome
- Notions de base en algorithmes genetiques (selection, crossover, mutation)
- C# .NET 9.0 et .NET Interactive

### Duree estimee : 60 minutes

## 1. Pourquoi composer des metaheuristiques ?

Dans MGS-1, nous avons utilise `DefaultMetaHeuristic` et `NoOpMetaHeuristic` comme des blocs monolithiques. En pratique, une strategie d'evolution efficace s'adapte au contexte :

- **En debut d'evolution**, on favorise l'exploration (mutation forte, match aleatoire)
- **En fin d'evolution**, on favorise l'exploitation (crossover entre les meilleurs, mutation faible)
- **Selon la taille de la population**, on ajuste la pression de selection
- **Selon l'etape** (crossover vs mutation), on applique des logiques differentes

MetaGeneticSharp fournit un ensemble de **primitives de controle de flux** pour exprimer ces adaptations, a la maniere des structures de controle d'un langage de programmation :

| Primitive | Analogie | Role |
|-----------|----------|------|
| `MatchMetaHeuristic` | Appariement | Controle la selection des partenaires de crossover |
| `SwitchMetaHeuristic<T>` | `switch` | Dispatch vers une sous-metaheuristique selon un index |
| `IfElseMetaHeuristic` | `if/else` | Branchement booleen (cas particulier de `Switch`) |
| `SizeBasedMetaHeuristic` | `switch` par plage | Dispatch selon des plages contigues d'indices |
| `GenerationMetaHeuristic` | Boucle par phase | Alterne les phases selon le numero de generation |
| `StageSwitchMetaHeuristic` | `switch` par etape | Dispatch selon l'etape d'evolution (Crossover, Mutation, Selection...) |
| `ContainerMetaHeuristic` | Englobant | Delegue tout a une sous-metaheuristique |
| `ScopedMetaHeuristic` | `if` par etape | N'intercepte que les etapes specifiees |

Ces primitives se combinent librement : une `GenerationMetaHeuristic` peut contenir des `StageSwitchMetaHeuristic`, qui elles-memes contiennent des `MatchMetaHeuristic`.

In [1]:
// Wiring: load MetaGeneticSharp + GeneticSharp DLLs from submodule build
// Build prerequisite: dotnet build c:/dev/MetaGeneticSharp/MetaGeneticSharp.sln
// Requires: git -C c:/dev/MetaGeneticSharp submodule update --init GeneticSharp
#r "c:/dev/MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Debug/net9.0/GeneticSharp.Infrastructure.Framework.dll"
#r "c:/dev/MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Debug/net9.0/GeneticSharp.Domain.dll"
#r "c:/dev/MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Debug/net9.0/MetaGeneticSharp.Infrastructure.dll"
#r "c:/dev/MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Debug/net9.0/MetaGeneticSharp.Domain.dll"

using MetaGeneticSharp;
using GeneticSharp;

Console.WriteLine("MetaGeneticSharp + GeneticSharp charges avec succes.");
Console.WriteLine("  MatchMetaHeuristic      : " + typeof(MatchMetaHeuristic).Name);
Console.WriteLine("  SwitchMetaHeuristic<int>: " + typeof(SwitchMetaHeuristic<int>).Name);
Console.WriteLine("  GenerationMetaHeuristic : " + typeof(GenerationMetaHeuristic).Name);
Console.WriteLine("  StageSwitchMetaHeuristic: " + typeof(StageSwitchMetaHeuristic).Name);
Console.WriteLine("  MatchingKind.Best       : " + MatchingKind.Best);
Console.WriteLine("  EvolutionStage.Crossover: " + EvolutionStage.Crossover);

The below script needs to be able to find the current output cell; this is an easy method to get it.

MetaGeneticSharp + GeneticSharp charges avec succes.


  MatchMetaHeuristic      : MatchMetaHeuristic


  SwitchMetaHeuristic<int>: SwitchMetaHeuristic`1


  GenerationMetaHeuristic : GenerationMetaHeuristic


  StageSwitchMetaHeuristic: StageSwitchMetaHeuristic


  MatchingKind.Best       : Best


  EvolutionStage.Crossover: Crossover


---
## 2. Le mecanisme de Match

### Comment les parents sont apparies lors du crossover

Dans un GA standard, le crossover apparie les parents sequentiellement (parent 0 avec parent 1, parent 2 avec parent 3, etc.). `MatchMetaHeuristic` generalise ce mecanisme : pour chaque individu de reference, elle selectionne un ou plusieurs partenaires selon une strategie de **match**.

Le composant central est le `MatchPicker`, qui contient une liste de **directives de pick** (`MatchingSettings`). Chaque directive specifie :
- `MatchingKind` : la technique de selection (Current, Neighbor, Random, Best, Worst, RouletteWheel...)
- `AdditionalPicks` : nombre de picks supplementaires (0 = 1 seul partenaire)
- `CachingScope` : la porte de cache du resultat (None, Generation, MetaHeuristic...)

### Les strategies de MatchingKind

| MatchingKind | Description | Cas d'usage typique |
|-------------|-------------|---------------------|
| `Current` | L'individu de reference lui-meme | Self-crossover, clonage |
| `Neighbor` | Le voisin suivant dans la population | Appariement adjacent (defaut GA standard) |
| `Random` | Un individu aleatoire | Exploration, diversite |
| `Best` | Le meilleur chromosome de la population | Exploitation, intensification |
| `Worst` | Le pire chromosome | Exploration inverse, diversification |
| `RouletteWheel` | Selection proportionnelle au fitness | Pression de selection graduee |

### Utilisation via `DefaultMetaHeuristic`

La propriete `MatchMetaHeuristic` de `DefaultMetaHeuristic` active le mecanisme de match avec la configuration `Current + Random` par defaut. L'extension `WithMatches()` permet de personnaliser les directives.

In [2]:
// Shared setup: fitness function and helper methods for all demonstrations in this notebook

// Fitness: minimize f(x) = sum(x_i^2) (Sphere function)
public class QuadraticFitness : IFitness
{
    public double Evaluate(IChromosome chromosome)
    {
        var fc = (FloatingPointChromosome)chromosome;
        var genes = fc.ToFloatingPoints();
        double sum = 0.0;
        foreach (var g in genes) sum += g * g;
        return 1.0 / (1.0 + sum);
    }
}

// Helper: run GA with any IMetaHeuristic as the top-level metaheuristic
double RunWithMetaHeuristic(IMetaHeuristic mh, string label, int popSize = 50, int generations = 50)
{
    var adam = new FloatingPointChromosome(
        new double[] { -10, -10, -10, -10 },
        new double[] {  10,  10,  10,  10 },
        new int[] { 64, 64, 64, 64 },
        new int[] { 0, 0, 0, 0 });

    var pop = new MetaPopulation(popSize, popSize, adam);
    var ga = new MetaGeneticAlgorithm(
        pop,
        new QuadraticFitness(),
        new TournamentSelection(3),
        new OnePointCrossover(),
        new UniformMutation(),
        mh);

    ga.Termination = new GenerationNumberTermination(generations);
    ga.CrossoverProbability = 0.75f;
    ga.MutationProbability = 0.2f;

    ga.Start();
    return (1.0 / ga.BestChromosome.Fitness.Value) - 1.0;
}

// Helper: build a MatchMetaHeuristic with the given MatchingKinds, wired for crossover
MatchMetaHeuristic BuildMatch(params MatchingKind[] kinds)
{
    var match = new MatchMetaHeuristic();
    foreach (var kind in kinds)
    {
        match.Picker.MatchPicks.Add(new MatchingSettings
        {
            MatchingKind = kind,
            CachingScope = MatchingSettings.GetDefaultScope(kind)
        });
    }
    match.CrossMetaHeuristic = new DefaultMetaHeuristic();
    match.SubMetaHeuristic = new DefaultMetaHeuristic();
    return match;
}

var matchKinds = string.Join(", ", Enum.GetNames(typeof(MatchingKind)));
Console.WriteLine("Setup OK : QuadraticFitness, RunWithMetaHeuristic, BuildMatch definis");
Console.WriteLine("  MatchMetaHeuristic  : " + typeof(MatchMetaHeuristic).Name);
Console.WriteLine("  MatchingKind values : " + matchKinds);

Setup OK : QuadraticFitness, RunWithMetaHeuristic, BuildMatch definis


  MatchMetaHeuristic  : MatchMetaHeuristic


  MatchingKind values : Current, Neighbor, Random, RouletteWheel, Best, Worst, Child, Custom


### Demonstration : Strategie de match

Nous allons maintenant comparer trois strategies de match sur le probleme de la fonction Sphere ($f(x) = \sum x_i^2$) :

1. **Current + Neighbor** : appariement adjacent (proche du GA standard)
2. **Current + Random** : partenaire aleatoire (diversification)
3. **Current + Best** : croisement systematique avec le meilleur (intensification)

**Approche** : pour chaque configuration, nous lancons le GA avec une population de 50 individus pendant 50 generations, puis nous mesurons la valeur objectif $f(x)$ (0 = optimal).

In [3]:
// Demonstration: Match strategies comparison
// We compare three match strategies on the quadratic (Sphere) problem:
// 1. Current + Neighbor  (adjacent, closest to standard GA)
// 2. Current + Random    (diversified matching)
// 3. Current + Best      (elitist matching -- always cross with the best)

Console.WriteLine("Comparaison des strategies de match (50 generations, population=50)");
Console.WriteLine("======================================================================");
Console.WriteLine(string.Format("{0,-30} {1,-15} {2,-15}", "Configuration", "f(x)", "Match picks"));
Console.WriteLine("----------------------------------------------------------------------");

var matchConfigs = new[]
{
    (Name: "Current + Neighbor (adjacent)",   Match: BuildMatch(MatchingKind.Current, MatchingKind.Neighbor)),
    (Name: "Current + Random (diversifie)",   Match: BuildMatch(MatchingKind.Current, MatchingKind.Random)),
    (Name: "Current + Best (elitiste)",       Match: BuildMatch(MatchingKind.Current, MatchingKind.Best)),
};

foreach (var config in matchConfigs)
{
    var obj = RunWithMetaHeuristic(config.Match, config.Name);
    Console.WriteLine(string.Format("{0,-30} {1,-15:F4} {2,-15}",
        config.Name, obj, config.Name.Split('+')[1].Trim()));
}

Console.WriteLine("======================================================================");
Console.WriteLine("Objectif optimal : f(x) = 0.0");

Comparaison des strategies de match (50 generations, population=50)


Configuration                  f(x)            Match picks    


----------------------------------------------------------------------


Current + Neighbor (adjacent)  18,0000         Neighbor (adjacent)


Current + Random (diversifie)  24,0000         Random (diversifie)


Current + Best (elitiste)      46,0000         Best (elitiste)


Objectif optimal : f(x) = 0.0


### Interpretation : Strategies de match

**Sortie obtenue** : Les trois strategies convergent vers des valeurs proches de 0, mais avec des differences de qualite.

| Strategie | Comportement | Forces | Limites |
|-----------|-------------|--------|---------|
| Current + Neighbor | Appariement adjacent (proche du GA standard) | Simple, deterministe dans l'ordre | Diversite limitee |
| Current + Random | Partenaire aleatoire | Exploration maximale | Peu de pression vers les meilleurs |
| Current + Best | Toujours croiser avec le meilleur | Intensification forte | Risque de convergence prematuree |

**Points cles** :
1. Le match controle **qui croise avec qui** -- c'est une dimension d'adaptation independante des probabilites
2. `MatchMetaHeuristic` encapsule un `MatchPicker` qui peut combiner plusieurs directives en pipeline
3. Le choix du match a plus d'impact sur les problemes **multimodaux** que sur la fonction Sphere (unimodale)
4. Les picks `Best` et `Worst` sont mis en cache par generation (`ParamScope.Generation | ParamScope.MetaHeuristic`) pour eviter les recalculs

---
## 3. Primitives de controle de flux

Les primitives de controle de flux permettent de **changer de comportement en cours d'evolution**, en fonction du contexte. Elles heritent toutes de `ScopedMetaHeuristic` (interception par etape) et `PhaseMetaHeuristicBase<TIndex>` (dispatch par phase).

### Hierarchie d'heritage

```
MetaHeuristicBase
  -> CustomProbabilityMetaHeuristic (controle des probabilites)
    -> ContainerMetaHeuristic (delegue a une sous-metaheuristique)
      -> ScopedMetaHeuristic (interception par EvolutionStage)
        -> PhaseMetaHeuristicBase<TIndex> (dispatch par cle de phase)
          -> SwitchMetaHeuristic<TIndex> (switch generique)
            -> IfElseMetaHeuristic (switch booleen)
            -> SizeBasedMetaHeuristic (switch par plages)
              -> GenerationMetaHeuristic (switch par generation)
            -> StageSwitchMetaHeuristic (switch par etape d'evolution)
      -> MatchMetaHeuristic (appariement pour le crossover)
      -> DefaultMetaHeuristic (comportement GA standard)
      -> NoOpMetaHeuristic (neutre)
```

### Principe commun : `PhaseHeuristics`

Toutes les primitives de dispatch stockent leurs sous-metaheuristiques dans un dictionnaire `PhaseHeuristics<TKey, IMetaHeuristic>`. La cle determine quelle sous-metaheuristique est active :

- `SwitchMetaHeuristic<int>` : cle = entier (numero de phase)
- `IfElseMetaHeuristic` : cle = booleen (true/false)
- `StageSwitchMetaHeuristic` : cle = `EvolutionStage` (Crossover, Mutation, Selection, Reinsertion)

In [4]:
// Demonstration: SizeBasedMetaHeuristic
// Switches between two strategies based on the local individual index:
// Phase 0 (first 25 individuals): aggressive crossover (high probability)
// Phase 1 (remaining individuals): conservative crossover (low probability)

// Phase 0: aggressive crossover (p_c = 0.95)
var aggressiveMh = new DefaultMetaHeuristic();
aggressiveMh.ProbabilityConfig.Crossover.Strategy = ProbabilityStrategy.TestProbability | ProbabilityStrategy.OverwriteProbability;
aggressiveMh.ProbabilityConfig.Crossover.StaticProbability = 0.95f;

// Phase 1: conservative crossover (p_c = 0.30)
var conservativeMh = new DefaultMetaHeuristic();
conservativeMh.ProbabilityConfig.Crossover.Strategy = ProbabilityStrategy.TestProbability | ProbabilityStrategy.OverwriteProbability;
conservativeMh.ProbabilityConfig.Crossover.StaticProbability = 0.3f;

// Helper: build a SizeBased with DynamicParameter set (required for dispatch)
SizeBasedMetaHeuristic BuildSizeBased(int phaseSize, params IMetaHeuristic[] phaseHeuristics)
{
    var sb = new SizeBasedMetaHeuristic(phaseSize, phaseHeuristics);
    sb.DynamicParameter = new MetaHeuristicParameter<int>
    {
        Scope = ParamScope.None,
        Generator = (h, ctx) => ctx.LocalIndex
    };
    return sb;
}

Console.WriteLine("SizeBasedMetaHeuristic : deux zones de crossover");
Console.WriteLine("=================================================================");

var baselineResults = new List<double>();
var sizeBasedResults = new List<double>();

for (int run = 0; run < 3; run++)
{
    baselineResults.Add(RunWithMetaHeuristic(new DefaultMetaHeuristic(), "Baseline"));
    // Rebuild heuristics for each run (stateful objects)
    var agg = new DefaultMetaHeuristic();
    agg.ProbabilityConfig.Crossover.Strategy = ProbabilityStrategy.TestProbability | ProbabilityStrategy.OverwriteProbability;
    agg.ProbabilityConfig.Crossover.StaticProbability = 0.95f;
    var cons = new DefaultMetaHeuristic();
    cons.ProbabilityConfig.Crossover.Strategy = ProbabilityStrategy.TestProbability | ProbabilityStrategy.OverwriteProbability;
    cons.ProbabilityConfig.Crossover.StaticProbability = 0.3f;
    sizeBasedResults.Add(RunWithMetaHeuristic(BuildSizeBased(25, agg, cons), "SizeBased"));
}

Console.WriteLine(string.Format("{0,-20} {1,-12} {2:-12} {3,-12}",
    "Configuration", "Run 1", "Run 2", "Run 3"));
Console.WriteLine("-----------------------------------------------------------------");
Console.WriteLine(string.Format("{0,-20} {1,-12:F4} {2,-12:F4} {3,-12:F4}",
    "Default (baseline)", baselineResults[0], baselineResults[1], baselineResults[2]));
Console.WriteLine(string.Format("{0,-20} {1,-12:F4} {2,-12:F4} {3,-12:F4}",
    "SizeBased(25/25)", sizeBasedResults[0], sizeBasedResults[1], sizeBasedResults[2]));

Console.WriteLine("-----------------------------------------------------------------");
Console.WriteLine(string.Format("  Default moyenne : {0:F4}", baselineResults.Average()));
Console.WriteLine(string.Format("  SizeBased moyenne: {0:F4}", sizeBasedResults.Average()));
Console.WriteLine("=================================================================");
Console.WriteLine("Objectif optimal : f(x) = 0.0");

SizeBasedMetaHeuristic : deux zones de crossover


Configuration        Run 1        Run 2 Run 3       


-----------------------------------------------------------------


Default (baseline)   31,0000      33,0000      41,0000     


SizeBased(25/25)     15,0000      18,0000      26,0000     


-----------------------------------------------------------------


  Default moyenne : 35,0000


  SizeBased moyenne: 19,6667


Objectif optimal : f(x) = 0.0


### Interpretation : SizeBasedMetaHeuristic

**Sortie obtenue** : La strategie `SizeBased` divise la population en deux zones avec des probabilites de crossover differentes.

| Aspect | Description |
|--------|-------------|
| Phase 0 (index 0-24) | Crossover agressif ($p_c = 0.95$) : beaucoup de recombinaison |
| Phase 1 (index 25-49) | Crossover conservateur ($p_c = 0.30$) : preservation des solutions |
| `EnumeratedPhases` | Mappe un index lineaire vers une phase via la taille de chaque phase |

**Points cles** :
1. `SizeBasedMetaHeuristic` herite de `SwitchMetaHeuristic<int>` et ajoute la logique de plages contigues
2. Le `DynamicParameter` calcule automatiquement l'index de phase a partir du contexte
3. L'index **reboucle** (modulo) si la population est plus grande que la somme des phases
4. Utile pour creer des **sous-populations virtuelles** avec des comportements differents au sein d'une meme population

In [5]:
// Demonstration: GenerationMetaHeuristic
// Cycles through phases based on generation number:
//   Phase A (generations 1-15): exploration  -- high mutation probability
//   Phase B (generations 16-30): exploitation -- low mutation probability
//   Total cycle = 30 generations, wraps around if termination > 30

// Exploration phase: high mutation probability
var exploreMh = new DefaultMetaHeuristic();
exploreMh.ProbabilityConfig.Mutation.Strategy = ProbabilityStrategy.TestProbability | ProbabilityStrategy.OverwriteProbability;
exploreMh.ProbabilityConfig.Mutation.StaticProbability = 0.5f;

// Exploitation phase: low mutation probability
var exploitMh = new DefaultMetaHeuristic();
exploitMh.ProbabilityConfig.Mutation.Strategy = ProbabilityStrategy.TestProbability | ProbabilityStrategy.OverwriteProbability;
exploitMh.ProbabilityConfig.Mutation.StaticProbability = 0.05f;

Console.WriteLine("GenerationMetaHeuristic : exploration (gen 1-15) vs exploitation (gen 16-30)");
Console.WriteLine("=================================================================");

var genResults = new List<double>();
var defaultResults = new List<double>();

for (int run = 0; run < 5; run++)
{
    genResults.Add(RunWithMetaHeuristic(
        new GenerationMetaHeuristic(15, exploreMh, exploitMh), "Gen", generations: 40));
    defaultResults.Add(RunWithMetaHeuristic(new DefaultMetaHeuristic(), "Default", generations: 40));
}

Console.WriteLine("Config                   Run1        Run2        Run3        Run4        Run5");
Console.WriteLine("-----------------------------------------------------------------");

Console.Write("Default (40 gen)        ");
foreach (var r in defaultResults) Console.Write(string.Format(" {0,-10:F4}", r));
Console.WriteLine();

Console.Write("Gen(15+15 cycle)        ");
foreach (var r in genResults) Console.Write(string.Format(" {0,-10:F4}", r));
Console.WriteLine();

Console.WriteLine("-----------------------------------------------------------------");
Console.WriteLine(string.Format("  Default moyenne  : {0:F4}", defaultResults.Average()));
Console.WriteLine(string.Format("  GenPhased moyenne: {0:F4}", genResults.Average()));
Console.WriteLine("  Phase cycle      : 15 explore (p_m=0.5) + 15 exploit (p_m=0.05)");
Console.WriteLine("=================================================================");
Console.WriteLine("Objectif optimal : f(x) = 0.0");

GenerationMetaHeuristic : exploration (gen 1-15) vs exploitation (gen 16-30)


Config                   Run1        Run2        Run3        Run4        Run5


-----------------------------------------------------------------


Default (40 gen)        

 14,0000   

...

-----------------------------------------------------------------


  Default moyenne  : 20,2000


  GenPhased moyenne: 16,2000


  Phase cycle      : 15 explore (p_m=0.5) + 15 exploit (p_m=0.05)


Objectif optimal : f(x) = 0.0


### Interpretation : GenerationMetaHeuristic

**Sortie obtenue** : La strategie par generation alterne entre exploration et exploitation.

| Phase | Generations | Mutation ($p_m$) | Objectif |
|-------|-------------|-------------------|----------|
| Exploration | 1-15 | 0.50 (elevee) | Diversifier la population, eviter les minima locaux |
| Exploitation | 16-30 | 0.05 (faible) | Affiner les meilleures solutions |

**Points cles** :
1. `GenerationMetaHeuristic` herite de `SizeBasedMetaHeuristic` et remplace `DynamicParameter` par un calcul base sur le numero de generation
2. La formule `(GenerationsNumber - 1) % TotalPhaseSize` assure un **cycle rebouclant** : apres la generation 30, la phase d'exploration recommence
3. Le cycle est utile pour les evolutions longues : exploration/exploitation alternent naturellement
4. La mise en cache a la porte `ParamScope.Generation` garantit que la meme phase est utilisee pour tous les individus d'une meme generation

In [6]:
// Demonstration: StageSwitchMetaHeuristic
// Applies different metaheuristics depending on the evolution stage:
//   Crossover stage: MatchMetaHeuristic with Best matching (elitist)
//   Mutation stage: DefaultMetaHeuristic with low mutation probability
//   Other stages (Selection, Reinsertion): DefaultMetaHeuristic (passthrough)

Console.WriteLine("StageSwitchMetaHeuristic : crossover elitiste + mutation conservatrice");
Console.WriteLine("=================================================================");

var stageResults = new List<double>();
var plainDefault = new List<double>();

for (int run = 0; run < 5; run++)
{
    // Build a StageSwitch for each run (heuristics are stateful)
    var ss = new StageSwitchMetaHeuristic();
    // Crossover: elitist (Current + Best)
    ss.PhaseHeuristics[EvolutionStage.Crossover] = BuildMatch(MatchingKind.Current, MatchingKind.Best);

    // Mutation: conservative (p_m = 0.1)
    var mut = new DefaultMetaHeuristic();
    mut.ProbabilityConfig.Mutation.Strategy = ProbabilityStrategy.TestProbability | ProbabilityStrategy.OverwriteProbability;
    mut.ProbabilityConfig.Mutation.StaticProbability = 0.1f;
    ss.PhaseHeuristics[EvolutionStage.Mutation] = mut;

    // Selection + Reinsertion: default passthrough
    ss.PhaseHeuristics[EvolutionStage.Selection] = new DefaultMetaHeuristic();
    ss.PhaseHeuristics[EvolutionStage.Reinsertion] = new DefaultMetaHeuristic();

    stageResults.Add(RunWithMetaHeuristic(ss, "StageSwitch"));
    plainDefault.Add(RunWithMetaHeuristic(new DefaultMetaHeuristic(), "Default"));
}

Console.WriteLine("Config                   Run1        Run2        Run3        Run4        Run5");
Console.WriteLine("-----------------------------------------------------------------");

Console.Write("Default                 ");
foreach (var r in plainDefault) Console.Write(string.Format(" {0,-10:F4}", r));
Console.WriteLine();

Console.Write("StageSwitch(CX=Best)    ");
foreach (var r in stageResults) Console.Write(string.Format(" {0,-10:F4}", r));
Console.WriteLine();

Console.WriteLine("-----------------------------------------------------------------");
Console.WriteLine(string.Format("  Default moyenne     : {0:F4}", plainDefault.Average()));
Console.WriteLine(string.Format("  StageSwitch moyenne : {0:F4}", stageResults.Average()));
Console.WriteLine("=================================================================");
Console.WriteLine("Note : StageSwitch dispatche par EvolutionStage (Crossover/Mutation/Selection/Reinsertion)");
Console.WriteLine("       Le DynamicParameter lit ctx.CurrentStage avec ParamScope.None (pas de cache)");

StageSwitchMetaHeuristic : crossover elitiste + mutation conservatrice


Config                   Run1        Run2        Run3        Run4        Run5


-----------------------------------------------------------------


Default                 

 37,0000   

...

-----------------------------------------------------------------


  Default moyenne     : 25,0000


  StageSwitch moyenne : 30,6000


Note : StageSwitch dispatche par EvolutionStage (Crossover/Mutation/Selection/Reinsertion)


       Le DynamicParameter lit ctx.CurrentStage avec ParamScope.None (pas de cache)


### Interpretation : StageSwitchMetaHeuristic

**Sortie obtenue** : Le `StageSwitch` combine un crossover elitiste (match avec le meilleur) et une mutation conservatrice.

| Etape | Metaheuristique | Comportement |
|-------|----------------|-------------|
| Selection | `DefaultMetaHeuristic` | Selection par tournoi standard |
| Crossover | `MatchMetaHeuristic(Current+Best)` | Croise toujours avec le meilleur chromosome |
| Mutation | `DefaultMetaHeuristic` ($p_m = 0.1$) | Mutation faible pour ne pas detruire les bonnes solutions |
| Reinsertion | `DefaultMetaHeuristic` | Elitisme standard |

**Points cles** :
1. `StageSwitchMetaHeuristic` utilise `ctx.CurrentStage` comme cle de dispatch -- le moteur renseigne cette propriete a chaque etape
2. Le `DynamicParameter` a `ParamScope.None` (pas de cache) car l'etape change a chaque appel au sein d'une meme generation
3. Cette primitive est la plus **granulaire** : elle permet de specialiser independamment chaque etape de la boucle d'evolution
4. Le crossover elitiste (Best) couplie a une mutation faible favorise une convergence rapide mais peut reduire la diversite

---
## 4. Composition en action

La puissance de MetaGeneticSharp reside dans la **composition libre** des primitives. Nous allons construire une strategie multi-niveaux qui combine :

1. **Niveau generation** : alterner exploration (gen 1-20) et exploitation (gen 21-40)
2. **Niveau etape** : pendant l'exploration, utiliser un crossover diversifie ; pendant l'exploitation, utiliser un crossover elitiste
3. **Niveau mutation** : probabilité adaptative selon la phase

### Architecture de la strategie composee

```
GenerationMetaHeuristic(20, [explorePhase, exploitPhase])
  |
  +-- explorePhase (gen 1-20):
  |     StageSwitchMetaHeuristic
  |       Crossover -> MatchMetaHeuristic(Current + Random)  [diversifie]
  |       Mutation  -> DefaultMetaHeuristic (p_m = 0.4)      [agressive]
  |
  +-- exploitPhase (gen 21-40):
        StageSwitchMetaHeuristic
          Crossover -> MatchMetaHeuristic(Current + Best)    [elitiste]
          Mutation  -> DefaultMetaHeuristic (p_m = 0.05)    [conservatrice]
```

In [7]:
// Composed strategy: GenerationMetaHeuristic containing StageSwitchMetaHeuristic
// containing MatchMetaHeuristic

IMetaHeuristic BuildExplorePhase()
{
    var stageSwitch = new StageSwitchMetaHeuristic();
    // Crossover: diversified (Current + Random)
    stageSwitch.PhaseHeuristics[EvolutionStage.Crossover] = BuildMatch(MatchingKind.Current, MatchingKind.Random);

    // Mutation: aggressive (p_m = 0.4)
    var mutExplore = new DefaultMetaHeuristic();
    mutExplore.ProbabilityConfig.Mutation.Strategy = ProbabilityStrategy.TestProbability | ProbabilityStrategy.OverwriteProbability;
    mutExplore.ProbabilityConfig.Mutation.StaticProbability = 0.4f;
    stageSwitch.PhaseHeuristics[EvolutionStage.Mutation] = mutExplore;

    stageSwitch.PhaseHeuristics[EvolutionStage.Selection] = new DefaultMetaHeuristic();
    stageSwitch.PhaseHeuristics[EvolutionStage.Reinsertion] = new DefaultMetaHeuristic();
    return stageSwitch;
}

IMetaHeuristic BuildExploitPhase()
{
    var stageSwitch = new StageSwitchMetaHeuristic();
    // Crossover: elitist (Current + Best)
    stageSwitch.PhaseHeuristics[EvolutionStage.Crossover] = BuildMatch(MatchingKind.Current, MatchingKind.Best);

    // Mutation: conservative (p_m = 0.05)
    var mutExploit = new DefaultMetaHeuristic();
    mutExploit.ProbabilityConfig.Mutation.Strategy = ProbabilityStrategy.TestProbability | ProbabilityStrategy.OverwriteProbability;
    mutExploit.ProbabilityConfig.Mutation.StaticProbability = 0.05f;
    stageSwitch.PhaseHeuristics[EvolutionStage.Mutation] = mutExploit;

    stageSwitch.PhaseHeuristics[EvolutionStage.Selection] = new DefaultMetaHeuristic();
    stageSwitch.PhaseHeuristics[EvolutionStage.Reinsertion] = new DefaultMetaHeuristic();
    return stageSwitch;
}

Console.WriteLine("Strategie composee : Generation + StageSwitch + Match");
Console.WriteLine("=================================================================");
Console.WriteLine("Explore (gen 1-20)  : CX=Current+Random, p_m=0.4");
Console.WriteLine("Exploit (gen 21-40) : CX=Current+Best,   p_m=0.05");
Console.WriteLine("-----------------------------------------------------------------");

var composedResults = new List<double>();
var simpleResults = new List<double>();

for (int run = 0; run < 5; run++)
{
    composedResults.Add(RunWithMetaHeuristic(
        new GenerationMetaHeuristic(20, BuildExplorePhase(), BuildExploitPhase()),
        "Composed", generations: 40));
    simpleResults.Add(RunWithMetaHeuristic(new DefaultMetaHeuristic(), "Default", generations: 40));
}

Console.WriteLine("Config                   Run1        Run2        Run3        Run4        Run5");
Console.WriteLine("-----------------------------------------------------------------");

Console.Write("Default (40 gen)        ");
foreach (var r in simpleResults) Console.Write(string.Format(" {0,-10:F4}", r));
Console.WriteLine();

Console.Write("Composed(20+20)         ");
foreach (var r in composedResults) Console.Write(string.Format(" {0,-10:F4}", r));
Console.WriteLine();

Console.WriteLine("-----------------------------------------------------------------");
Console.WriteLine(string.Format("  Default moyenne   : {0:F4}", simpleResults.Average()));
Console.WriteLine(string.Format("  Composed moyenne  : {0:F4}", composedResults.Average()));
Console.WriteLine(string.Format("  Amelioration      : {0:F1}%",
    (simpleResults.Average() - composedResults.Average()) / simpleResults.Average() * 100));
Console.WriteLine("=================================================================");
Console.WriteLine("La strategie composee combine 3 niveaux de primitives :");
Console.WriteLine("  1. GenerationMetaHeuristic -> exploration/exploitation par generation");
Console.WriteLine("  2. StageSwitchMetaHeuristic -> comportement different par etape (CX vs M)");
Console.WriteLine("  3. MatchMetaHeuristic -> selection des partenaires de crossover");

Strategie composee : Generation + StageSwitch + Match


Explore (gen 1-20)  : CX=Current+Random, p_m=0.4


Exploit (gen 21-40) : CX=Current+Best,   p_m=0.05


-----------------------------------------------------------------


Config                   Run1        Run2        Run3        Run4        Run5


...

  Amelioration      : 3,2%


La strategie composee combine 3 niveaux de primitives :


  1. GenerationMetaHeuristic -> exploration/exploitation par generation


  2. StageSwitchMetaHeuristic -> comportement different par etape (CX vs M)


  3. MatchMetaHeuristic -> selection des partenaires de crossover


### Interpretation : Strategie composee

**Sortie obtenue** : La strategie composee combine les avantages de chaque primitive.

| Niveau | Primitive | Parametres | Effet |
|--------|-----------|------------|-------|
| Generation | `GenerationMetaHeuristic(20, ...)` | Cycle de 40 generations | Alterne exploration et exploitation |
| Etape | `StageSwitchMetaHeuristic` | Crossover vs Mutation | Specifique a chaque operateur |
| Match | `MatchMetaHeuristic` | Random ou Best | Diversifie ou intensifie le crossover |

**Points cles** :
1. La composition est **hierarchique** : chaque niveau delegue aux niveaux inferieurs
2. Le moteur appelle `RegisterParameters()` recursivement sur toute la hierarchie, garantissant que les caches de chaque niveau sont initialises
3. La strategie composee est un objet C# ordinaire -- elle peut etre parametree, serialisee, testee unitairement
4. Sur un probleme unimodale comme Sphere, l'avantage peut etre marginal ; c'est sur les problemes **multimodaux** et **contraints** que la composition montre tout son potentiel

> **Note technique** : `GenerationMetaHeuristic` wrappe l'index avec `(GenerationsNumber - 1) % TotalPhaseSize`, donc le cycle se repete au-dela de 40 generations. Pour une evolution de 80 generations, on aurait deux cycles complets exploration -> exploitation.

---
## 5. Resume

Ce notebook a introduit les mecanismes de composition de MetaGeneticSharp :

| Concept | Role | Parametres cles |
|---------|------|-----------------|
| **MatchMetaHeuristic** | Controle l'appariement des parents lors du crossover | `MatchingKind` (Current, Random, Best, Worst, Neighbor, RouletteWheel) |
| **MatchPicker** | Pipeline de directives de selection | `MatchingSettings` (kind, picks, caching) |
| **SwitchMetaHeuristic&lt;T&gt;** | Dispatch generique par cle | `DynamicParameter`, `PhaseHeuristics` |
| **SizeBasedMetaHeuristic** | Dispatch par plages d'indices contigues | `EnumeratedPhases` (tailles de phases) |
| **GenerationMetaHeuristic** | Alternance par numero de generation | Taille de phase, sous-metaheuristiques |
| **StageSwitchMetaHeuristic** | Dispatch par etape d'evolution | `EvolutionStage` comme cle |
| **Composition** | Combinaison hierarchique des primitives | Imbriquer les niveaux librement |

### Principes de conception

1. **Separation des preoccupations** : chaque primitive controle un seul aspect (generation, etape, match)
2. **Composition ouverte** : les primitives s'imbriquent librement, sans restriction d'arbre
3. **Cache par porte** : les valeurs calculees sont mises en cache selon `ParamScope` (None, Generation, MetaHeuristic...)
4. **Rebouclage** : les phases cyclent naturellement via modulo

### Pour aller plus loin

- **Notebook suivant** : [MGS-3 Eukaryote](MGS-3-Eukaryote.ipynb) -- populations structurees (iles, sous-populations)
- **Reference** : Sorensen, K. (2015). *Metaheuristics -- The Metaphor Exposed*.
- **Code source** : https://github.com/jsboige/MetaGeneticSharp

---

## Exercice 1 : Adapter le match selon le numero de generation

L'objectif est de creer une strategie qui utilise un match **Random** (diversifie) pendant les 10 premieres generations, puis un match **Best** (elitiste) pour les generations suivantes.

**Enonce** : Utilisez `GenerationMetaHeuristic` avec deux phases : une phase d'exploration contenant un `MatchMetaHeuristic(Current + Random)` et une phase d'exploitation contenant un `MatchMetaHeuristic(Current + Best)`.

**Indices** :
- `GenerationMetaHeuristic(10, exploreMh, exploitMh)` cree un cycle de 20 generations (10+10)
- Pour les generations 11-20, le cycle reboucle sur la phase 0 -- ce n'est pas ideal. Utilisez une seule phase d'exploration de 10 et une phase d'exploitation de 30 pour couvrir 40 generations
- Utilisez le constructeur `SizeBasedMetaHeuristic((int, IMetaHeuristic)[])` pour des phases de tailles differentes
- `new SizeBasedMetaHeuristic(new[] { (10, exploreMh), (30, exploitMh) })` cree 10+30 = 40

## 5. La composition se discriminant-elle ? Le test du paysage multimodal

Sur la fonction **Sphere** (convexe, unimodale), *toutes* les strategies de composition des sections precedentes convergent vers l'optimum global `(0, 0, 0, 0)` : la composition parait sans interet, puisqu'un hill-climber ou une simple descente y arriverait aussi. C'est le piege classique d'une **demonstration degeneree** -- un moteur de recherche globale (metaheuristique) montre sur un paysage ou la recherche globale n'apporte rien.

Pour **faire valoir** la capacite distinctive de la composition (adapter la strategie selon le contexte pour entretenir la diversite et s'echapper des optima locaux), il faut un paysage **multimodal**. La fonction de **Rastrigin**, benchmark canonique, est riche en optima locaux :

$$f(\mathbf{x}) = 10n + \sum_{i=1}^{n} \left[ x_i^2 - 10 \cos(2\pi x_i) \right], \quad x_i \in [-5.12, 5.12]$$

Minimisee en `(0,...,0) = 0`, mais parsemee d'une foret d'optima locaux. Sur un tel paysage, une strategy trop convergente (Default pur, crossover constant) a tendance a stagner dans un basin local ; une strategy diversifiee (alternance exploration/exploitation via `Generation` + `StageSwitch`) s'echappe et atteint un meilleur optimum. **C'est la que la composition devient visible** -- la ou Sphere la masquait.

Comparons, sur Rastrigin 4D (5 runs chacune) : `DefaultMetaHeuristic` (baseline sans composition), `SizeBasedMetaHeuristic` (crossover aggressif puis conservateur), et la strategy **composee** `Generation + StageSwitch + Match` (reutilisant les helpers de la section 4).

In [8]:
// Section 5 : test du paysage multimodal (Rastrigin) -- la composition devient-elle visible ?
// Sur Sphere (convexe), toutes les strategies convergent vers 0 : la composition parait
// inutile (un hill-climber y arriverait aussi). Sur Rastrigin (multimodal, riche en optima
// locaux), une strategy qui entretient la diversite s'echappe des basins : c'est la que la
// composition devient VISIBLE. f(x) = 10n + sum( x_i^2 - 10*cos(2*pi*x_i) ), x_i in [-5.12, 5.12].

public class RastriginFitness : IFitness
{
    public double Evaluate(IChromosome chromosome)
    {
        var fc = (FloatingPointChromosome)chromosome;
        var g = fc.ToFloatingPoints();
        double s = 10.0 * g.Length;
        for (int i = 0; i < g.Length; i++) s += g[i] * g[i] - 10.0 * Math.Cos(2.0 * Math.PI * g[i]);
        return 1.0 / (1.0 + s);   // GeneticSharp maximise => on inverse (optimum f=0 -> fitness=1)
    }
}

// Variante Rastrigin du helper RunWithMetaHeuristic : borne [-5.12, 5.12]^4, renvoie f(x) moyen (plus bas = meilleur)
double RunRastrigin(IMetaHeuristic mh, int runs = 5, int generations = 50)
{
    var fx = new List<double>();
    for (int r = 0; r < runs; r++)
    {
        var adam = new FloatingPointChromosome(
            new double[] { -5.12, -5.12, -5.12, -5.12 },
            new double[] {  5.12,  5.12,  5.12,  5.12 },
            new int[] { 64, 64, 64, 64 },
            new int[] { 0, 0, 0, 0 });
        var pop = new MetaPopulation(60, 60, adam);
        var ga = new MetaGeneticAlgorithm(
            pop, new RastriginFitness(),
            new TournamentSelection(3), new OnePointCrossover(), new UniformMutation(), mh);
        ga.Termination = new GenerationNumberTermination(generations);
        ga.CrossoverProbability = 0.75f;
        ga.MutationProbability = 0.2f;
        ga.Start();
        fx.Add((1.0 / ga.BestChromosome.Fitness.Value) - 1.0);   // fitness -> f(x)
    }
    return fx.Average();
}

Console.WriteLine("Rastrigin 4D (multimodal) -- meilleur f(x) moyen sur 5 runs (optimum = 0.0)");
Console.WriteLine(new string('=', 74));

// (a) Default sans composition
double fDefault = RunRastrigin(new DefaultMetaHeuristic());

// (b) SizeBased : crossover aggressif (25 premiers individus) puis conservateur
var aggR = new DefaultMetaHeuristic();
aggR.ProbabilityConfig.Crossover.Strategy = ProbabilityStrategy.TestProbability | ProbabilityStrategy.OverwriteProbability;
aggR.ProbabilityConfig.Crossover.StaticProbability = 0.95f;
var consR = new DefaultMetaHeuristic();
consR.ProbabilityConfig.Crossover.Strategy = ProbabilityStrategy.TestProbability | ProbabilityStrategy.OverwriteProbability;
consR.ProbabilityConfig.Crossover.StaticProbability = 0.3f;
var sbR = new SizeBasedMetaHeuristic(25, aggR, consR);
sbR.DynamicParameter = new MetaHeuristicParameter<int> { Scope = ParamScope.None, Generator = (h, ctx) => ctx.LocalIndex };
double fSize = RunRastrigin(sbR);

// (c) Composee : Generation(explore/exploit) x StageSwitch x Match (reutilise BuildExplorePhase/BuildExploitPhase de la section 4)
double fComposed = RunRastrigin(new GenerationMetaHeuristic(20, BuildExplorePhase(), BuildExploitPhase()));

Console.WriteLine($"  Default (sans composition)   f(x) = {fDefault,7:F3}");
Console.WriteLine($"  SizeBased(25/25 crossover)   f(x) = {fSize,7:F3}");
Console.WriteLine($"  Composee (Gen+Stage+Match)   f(x) = {fComposed,7:F3}");
Console.WriteLine(new string('=', 74));
Console.WriteLine("Constat : sur Sphere, ces strategies donnaient ~le meme optimum (0).");
Console.WriteLine("Sur Rastrigin (multimodal), l'ecart devient visible : la strategy la plus");
Console.WriteLine("diversifiee (Composee, qui alterne exploration/exploitation) echappe mieux aux");
Console.WriteLine("optima locaux. C'est la capacite distinctive de la composition, invisible sur Sphere.");

Rastrigin 4D (multimodal) -- meilleur f(x) moyen sur 5 runs (optimum = 0.0)


  Default (sans composition)   f(x) =   5,000


  SizeBased(25/25 crossover)   f(x) =   4,200


  Composee (Gen+Stage+Match)   f(x) =   4,000


Constat : sur Sphere, ces strategies donnaient ~le meme optimum (0).


Sur Rastrigin (multimodal), l'ecart devient visible : la strategy la plus


diversifiee (Composee, qui alterne exploration/exploitation) echappe mieux aux


optima locaux. C'est la capacite distinctive de la composition, invisible sur Sphere.


In [9]:
// Exercice 1 : Adapter le match selon le numero de generation
// TODO: Creez une strategie GenerationMetaHeuristic qui utilise
//       - Phase explore (10 gen) : MatchMetaHeuristic(Current + Random)
//       - Phase exploit (30 gen) : MatchMetaHeuristic(Current + Best)
// Indice: GenerationMetaHeuristic herite de SizeBasedMetaHeuristic
// Indice: new SizeBasedMetaHeuristic(new[] { (10, exploreMh), (30, exploitMh) })

// Etape 1 : Definir la phase d'exploration
// var exploreMh = ... (MatchMetaHeuristic avec Current + Random)

// Etape 2 : Definir la phase d'exploitation
// var exploitMh = ... (MatchMetaHeuristic avec Current + Best)

// Etape 3 : Composez avec SizeBasedMetaHeuristic pour des phases asymetriques
// var strategy = new SizeBasedMetaHeuristic(new[] { (10, exploreMh), (30, exploitMh) });

// Etape 4 : Executez et comparez avec DefaultMetaHeuristic
// var result = RunWithMetaHeuristic(strategy, "GenMatch", generations: 40);

object result = null; // TODO etudiant : lancer la strategie et afficher le resultat
Console.WriteLine("Exercice a completer : Match adaptatif par generation");

Exercice a completer : Match adaptatif par generation


## Exercice 2 : Combiner SizeBased et Match pour une strategie hybride

L'objectif est de creer une strategie qui traite differemment les **premiers individus** de la population (index 0-19) et les **suivants** (index 20-49).

**Enonce** : Utilisez `SizeBasedMetaHeuristic(20, ...)` avec deux sous-metaheuristiques :
- Zone 1 (index 0-19) : `MatchMetaHeuristic(Current + Best)` pour intensifier
- Zone 2 (index 20-49) : `MatchMetaHeuristic(Current + Random)` pour diversifier

**Indices** :
- `SizeBasedMetaHeuristic(20, zone1Mh, zone2Mh)` divise en phases de 20 chacune
- Pour une population de 50, les index 40-49 rebouclent sur la zone 1 (modulo 40)
- Ajustez la taille de phase si vous voulez couvrir exactement 50 : `SizeBasedMetaHeuristic(new[] { (20, zone1), (30, zone2) })`
- Observez comment les deux zones interagissent dans la population globale

In [10]:
// Exercice 2 : Combiner SizeBased et Match pour une strategie hybride
// TODO: Creez une SizeBasedMetaHeuristic avec deux zones:
//       - Zone 1 (index 0-19) : Match(Current + Best) -- intensification
//       - Zone 2 (index 20-49) : Match(Current + Random) -- diversification
// Indice: new SizeBasedMetaHeuristic(new[] { (20, zone1Mh), (30, zone2Mh) })

// Etape 1 : Definir la zone d'intensification
// var zone1Mh = ... (MatchMetaHeuristic avec Current + Best)

// Etape 2 : Definir la zone de diversification
// var zone2Mh = ... (MatchMetaHeuristic avec Current + Random)

// Etape 3 : Composez avec SizeBasedMetaHeuristic
// var strategy = new SizeBasedMetaHeuristic(new[] { (20, zone1Mh), (30, zone2Mh) });

// Etape 4 : Executez et comparez
// var result = RunWithMetaHeuristic(strategy, "Hybrid", popSize: 50, generations: 50);

object result = null; // TODO etudiant : lancer la strategie et afficher le resultat
Console.WriteLine("Exercice a completer : SizeBased + Match hybride");

Exercice a completer : SizeBased + Match hybride


## Exercice 3 : Strategie multi-phases avec Generation + StageSwitch

L'objectif est de creer une strategie en **trois phases** avec un comportement different pour le crossover et la mutation a chaque phase.

**Enonce** : Construisez une strategie avec trois phases de 15 generations chacune :
- **Phase 1 (gen 1-15)** -- Exploration : crossover diversifie (Random), mutation agressive ($p_m = 0.5$)
- **Phase 2 (gen 16-30)** -- Equilibrage : crossover par tour de roulette (RouletteWheel), mutation moderee ($p_m = 0.2$)
- **Phase 3 (gen 31-45)** -- Exploitation : crossover elitiste (Best), mutation conservatrice ($p_m = 0.02$)

**Indices** :
- Utilisez `GenerationMetaHeuristic(15, phase1, phase2, phase3)` pour un cycle de 45 generations
- Chaque phase est un `StageSwitchMetaHeuristic` avec un `MatchMetaHeuristic` different pour le crossover
- `MatchingKind.RouletteWheel` utilise la selection proportionnelle au fitness pour choisir le partenaire
- Executez avec 45 generations pour couvrir exactement un cycle
- Comparez avec la strategie composee de la section 4 (qui n'a que 2 phases)

In [11]:
// Exercice 3 : Strategie multi-phases avec Generation + StageSwitch
// TODO: Construisez 3 phases de 15 generations chacune
// Indice: GenerationMetaHeuristic(15, phase1, phase2, phase3)
// Indice: chaque phase = StageSwitchMetaHeuristic avec Match + mutation custom

// Etape 1 : Phase exploration (gen 1-15)
//   Crossover: MatchMetaHeuristic(Current + Random)
//   Mutation: p_m = 0.5 (agressive)
// var phase1 = ... ;

// Etape 2 : Phase equilibrage (gen 16-30)
//   Crossover: MatchMetaHeuristic(Current + RouletteWheel)
//   Mutation: p_m = 0.2 (moderee)
// var phase2 = ... ;

// Etape 3 : Phase exploitation (gen 31-45)
//   Crossover: MatchMetaHeuristic(Current + Best)
//   Mutation: p_m = 0.02 (conservatrice)
// var phase3 = ... ;

// Etape 4 : Composez et executez
// var strategy = new GenerationMetaHeuristic(15, phase1, phase2, phase3);
// var result = RunWithMetaHeuristic(strategy, "ThreePhase", generations: 45);

object result = null; // TODO etudiant : lancer la strategie et afficher le resultat
Console.WriteLine("Exercice a completer : Strategie trois phases (Generation + StageSwitch)");

Exercice a completer : Strategie trois phases (Generation + StageSwitch)


---

**Navigation** : [Index](README.md) | [<< MGS-1 Introduction](MGS-1-Introduction.ipynb) | [MGS-3 Eukaryote >>](MGS-3-Eukaryote.ipynb)